# 2. 텍스트 데이터 다루기

## 2.1. Text Tokenization

In [ ]:
import urllib.request

url = ("https://raw.githubusercontent.com/rickiepark/"
      "llm-from-scratch/main/ch02/01_main-chapter-code/"
      "the-verdict.txt")
file_path = "the-verdict.txt"
urllib.request.urlretrieve(url, file_path)

In [ ]:
# 단편 소설을 text sample로 읽기
# 실제 LLM 훈련 시 수백만 개의 글과 수십만 개의 책을 처리하는 게 일반적임
with open("the-verdict.txt", "r", encoding="utf-8") as f :
    raw_text = f.read()

print("총 문자 개수: ", len(raw_text))
print(raw_text[:99])


### 1-1. 정규표현식 적용 (이후 Tokenizer 적용 예정)

In [ ]:
import re

preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [token for token in preprocessed if token.strip()] # 중복된 공백 제거

print(len(preprocessed))
print(preprocessed[:30])

## 2.2. Convert tokens to token IDs

#### 토큰 ID를 임베딩 벡터로 변환하기 전 중간 단계
#### 토큰 ID로 매핑하려면 어휘사전 먼저 구축할 필요가 있음

In [ ]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print("총 단어 개수: ", vocab_size)
print("단어 목록: ", all_words[:30])

##### 어휘사전 구축

In [ ]:
# 첫 번째 줄 코드 설명
# enumerate() 함수는 반복 가능한 객체(예: 리스트, 튜플 등)를 인덱스와 함께 반환하는 함수
# 여기서는 vocab.items()를 enumerate()로 감싸서 각 단어와 그에 해당하는 정수 인덱스를 함께 반환
# token:integer for integer, token in enumerate(all_words) 설명 => all_words 리스트의 각 단어(token)에 대해, enumerate()가 반환하는 정수(integer)를 사용하여 vocab 딕셔너리를 생성
myVocab = {token:integer for integer, token in enumerate(all_words)} # vocab 딕셔너리는 각 단어(token)를 키로, 해당 단어의 정수 인덱스(integer)를 값으로 가지는 딕셔너리

for i, item in enumerate(myVocab.items()):
    print(i, item)
    if i >= 50:
        break

#### 이제 어휘사전을 새로운 텍스트에 적용하여 토큰 ID로 변환

#### 간단한 text tokenizer
- encode 메서드: 텍스트를 토큰으로 분할 + 어휘사전으로 문자열-정수 매핑을 수행 -> 토큰 ID 생성
- decode 메서드: 토큰 ID를 텍스트로 변환 (역방향 정수-문자열 매핑)

In [ ]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        # encode 메서드와 decode 메서드에서 참조할 수 있도록 어휘사전을 클래스 속성으로 저장
        self.str_to_int = vocab 
        
        # 토큰 ID를 원본 텍스트 토큰으로 매핑하는 역어휘사전 생성
        self.int_to_str = {i:s for s, i in vocab.items()}
        
    def encode(self, text):     # 입력 텍스트를 처리하여 토큰 ID로 변환
        preprocessed = re.split(r'([,.?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
    
    def decode(self, ids):      # 토큰 ID를 원본 텍스트로 되돌림
        text = " ".join([self.int_to_str[i] for i in ids])
        # 지정된 구두점 문자 앞의 공백을 삭제함
        text = re.sub(r'\s+([,.?_!"()\'])', r'\1', text)
        return text        

In [ ]:
tokenizer = SimpleTokenizerV1(myVocab)
text = """ "It's the last he painted, you know,"
       Mrs.Gisburn said with pardonable pride. """
ids = tokenizer.encode(text)
print("인코딩된 토큰 ID: ", ids)
text_decoded = tokenizer.decode(ids)
print("디코딩된 텍스트: ", text_decoded)

#### 훈련 세트에 없는 새로운 텍스트 샘플 적용

In [ ]:
text = "Hello, do you like tea?"

# 오류 발생 -> 소설 [The Verdict]에 "Hello" 라는 단어가 없기 때문에 어휘사전에 "Hello"가 포함되어 있지 않음
# LLM 훈련 시에는 대용량의 훈련 세트가 필요함
print("인코딩된 토큰 ID: ", tokenizer.encode(text))

## 2.3. Adding Special Context Tokens 
#### 알지 못하는 단어를 처리하려면 tokenizer 수정 필요
#### 모델이 텍스트에서 문맥이나 그 외 정보를 잘 이해하도록 특수 문맥 토큰 추가
- 모르는 단어, 문서 경계 등 표시
- <|unk|> : 어휘사전에 없는, 모르는 단어
- <|endoftext|>: 관련이 없는 텍스트 사이
    - 여러 개의 독립적인 문서/책으로 LLM 훈련 시 이전 텍스트 소스 다음 등장하는 문서/책 앞에 해당 토큰 추가

#### 고유 단어 목록에 특수 토큰을 추가하여 어휘사전 수정

In [ ]:
all_tokens = sorted(set(preprocessed))
all_tokens.extend(["<|endoftext|>","<|unk|>"])
myVocab2 = {token:integer for integer, token in enumerate(all_tokens)}

# 이전에는 총 단어 개수가 1,130이었음
print("총 단어 개수: ", len(myVocab2.items()))

In [ ]:
for i, item in enumerate(list(myVocab2.items())[-5:]):
    print(i, item)

#### 모르는 단어를 처리하는 간단한 text tokenizer 

In [ ]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        # encode 메서드와 decode 메서드에서 참조할 수 있도록 어휘사전을 클래스 속성으로 저장
        self.str_to_int = vocab 
        
        # 토큰 ID를 원본 텍스트 토큰으로 매핑하는 역어휘사전 생성
        self.int_to_str = {i:s for s, i in vocab.items()}
        
    def encode(self, text):     # 입력 텍스트를 처리하여 토큰 ID로 변환
        preprocessed = re.split(r'([,.?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        
        # 모르는 단어를 <|unk|> 토큰으로 대체 (vocab에 없는 단어는 전부 "<|unk|>"로 바뀜)
        preprocessed = [item if item in self.str_to_int 
                        else "<|unk|>" for item in preprocessed]
        
        # 여기서 "<|unk|>"를 ID로 변환해야 함
        ids = [self.str_to_int[s] for s in preprocessed]
        
        return ids
    
    def decode(self, ids):      # 토큰 ID를 원본 텍스트로 되돌림
        text = " ".join([self.int_to_str[i] for i in ids])
        # 구두점 문자 앞의 공백을 삭제
        text = re.sub(r'\s+([,.?_!"()\'])', r'\1', text)
        return text        

In [ ]:
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join([text1, text2])

print(text) 


In [ ]:
tokenizer = SimpleTokenizerV2(myVocab2)

# <|endoftext|> 토큰에 해당하는 1130과 <|unk|> 토큰에 해당하는 1131(Hello, palace)이 포함된 토큰 ID 리스트가 출력됨
print(tokenizer.encode(text))
print(tokenizer.decode(tokenizer.encode(text)))

#### LLM에 따라 추가적인 특수 토큰을 사용하기도 함
• [BOS](beginning of sequence) – 이 토큰은 텍스트의 시작을 표시합니다. 즉, 콘텐츠의 시작 부분을 LLM에 알려줍니다.

• [EOS](end of sequence) – 이 토큰은 텍스트 끝에 위치하며 <|endoftext|>와 비슷하게 관련이 없는 여러 개의 텍스트를 연결할 때 유용합니다. 예를 들어 서로 다른 2개의 위키백과 문서나 책을 합칠 때 [EOS] 토큰이 문서 하나가 끝나고 다음 문서가 시작되는 위치를 나타냅니다.

• [PAD] – 하나 이상의 배치 크기로 LLM을 훈련할 때 배치 안에 길이가 다른 텍스트가 포함될 수 있습니다. 모든 텍스트의 길이를 동일하게 맞추기 위해 짧은 텍스트를 [PAD] 토큰을 사용해 배치에서 가장 긴 텍스트의 길이까지 확장 또는 ‘패딩(padding)’합니다.

#### GPT에서 사용하는 토크나이저 : <|unk|> 토큰도 사용 x

## 2.4. Byte Pair Encoding (BPE)

#### 단어를 부분단어로 분할하는 바이트 페어 인코딩 (BPE) 토크나이저 사용
#### 오픈소스 라이브러리 (tiktoken) 사용
- https://github.com/openai/tiktoken

In [ ]:
!pip install tiktoken

In [ ]:
from importlib.metadata import version
import tiktoken
print("tiktoken 버전: ", version("tiktoken"))

#### BPE Tokenizer initialize

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
    " of someunknownPlace"
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

In [ ]:
strings = tokenizer.decode(integers)
print(strings)

In [ ]:
string_example = tokenizer.decode(tokenizer.encode(text, allowed_special={"<|endoftext|>"})[-3:])
string_example2 = tokenizer.decode([617, 34680, 27271])
print(string_example)
print(string_example2)

#### 주요 관찰점
##### 1. <|endoftext|> 토큰은 50256과 같이 비교적 큰 토큰 ID에 할당됨
    사실 GPT2, GPT3 등 ChatGPT 초기 모델 훈련 시에 사용된 BPE Tokenizer는 50,257 크기의 어휘사전을 가지고 있음 -> <|endoftext|>에 가장 큰 토큰
##### 2. BPE Tokenizer는 someunknownplace와 같은 알지 못하는 단어를 정확하게 인코딩하고 디코딩함
    -> BPE는 모르는 어떤 단어라도 처리할 수 있음 (<|unk|> 토큰 없이 어떻게?)

#### BPE 알고리즘
* 어휘사전에 없는 단어를 더 작은 부분단어, 심지어 개별 문자로 나누어 처음 단어를 처리함
* 반복적으로 자주 등장하는 문자를 부분적으로 합치고 다시 자주 등장하는 부분단어를 단어로 합쳐서 어휘사전을 구축함
    - ex) BPE는 먼저 모든 개별 문자("a", "b" 등)를 어휘사전에 추가함
    - 다음 단계에서 자주 등장하는 문자 조합을 부분단어로 합침



#### 연습문제 2.1

In [ ]:
text_practice = "Akwirw ier"

integers_practice = tokenizer.encode(text_practice, allowed_special={"<|endoftext|>"})
print(integers_practice)

for _, item in enumerate(integers_practice):
    print(item, tokenizer.decode([item]))

## 2.5. Data Sampling Using Sliding Window

#### LLM을 위한 embedding을 만드는 단계 : LLM 훈련에 필요한 input-target pair 생성
#### Sliding Window를 사용하여 train dataset에서 input-target pair를 추출하는 data loader 구현

#### BPE Tokenizer로 [The Verdict] 전체 토큰화

In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f :
    raw_text = f.read()
    
encoded_text = tokenizer.encode(raw_text)
print("인코딩된 텍스트의 길이: ", len(encoded_text))

#### 조금 더 흥미로운 텍스트 구절을 만들기 위해 첫 50개 토큰 삭제

In [ ]:
encoded_sample = encoded_text[50:]
print(encoded_sample[:20])

#### 다음 단어 예측 작업을 위해 input-target pair을 만드는 가장 직관적인 방법
    - input token을 담은 x와 input에서 token 하나만큼 이동한 타깃을 담은 y 변수를 만드는 것

In [ ]:
# 문맥 크기는 입력에 얼마나 많은 토큰을 포함할지 결정함
context_size = 4
x = encoded_sample[:context_size]
y = encoded_sample[1:context_size+1]

print(f"x: {x}")
print(f"y: {y}")

In [ ]:
for i in range(1, context_size+1):
    context = encoded_sample[:i]
    desired = encoded_sample[i]
    print(f"문맥: {context} -> 다음 토큰: {desired}")

In [ ]:
# LLM 훈련을 위한 input-target 쌍 생성 완료
for i in range(1, context_size+1):
    context = encoded_sample[:i]
    desired = encoded_sample[i]
    print(f"문맥: {tokenizer.decode(context)} -> 다음 토큰: {tokenizer.decode([desired])}")

#### 효율적인 dataloader 구현 
    - data loader : input dataset을 순회하며 pytorch tensor로 input과 target을 반환
    - 그림과 달리 코드 구현은 토큰 ID 사용 (BPE 토크나이저의 encode 메서드가 토큰화 + 토큰 ID 변환을 모두 한 번에 처리하기 때문임)


#### 배치 입력과 타겟을 위한 데이터셋 클래스
    - pytorch 데이터셋 클래스를 기반으로, 데이터셋에서 개별 행을 추출하는 방법 정의
    - 각 행은 input_chunk 텐서에 할당된(max_length만큼) 여러 개의 토큰 ID로 구성됨
    - target_chunk 텐서는 각 행에 상응하는 target을 가지고 있음

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader   

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride) :
        self.input_ids = []
        self.target_ids = []
        
        # 전체 텍스트를 토큰화함
        token_ids = tokenizer.encode(txt)
        
        # 슬라이딩 윈도우를 사용해 문서를 max_length 길이의 중첩된 시퀀스로 나눔
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i+max_length]
            target_chunk = token_ids[i+1:i+max_length+1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    # 데이터셋에 있는 전체 행 수를 반환하는 메서드
    def __len__(self):
        return len(self.input_ids)
    
    # 데이터셋에서 하나의 행을 반환하는 메서드
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

#### input-target 쌍의 배치를 생성하기 위한 data loader

In [ ]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                      stride=128, shuffle=True, drop_last=True,
                      num_workers=0):
    # 토크나이저 초기화
    tokenizer = tiktoken.get_encoding("gpt2")
    
    # GPTDatasetV1 데이터셋 인스턴스 생성
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        
        # drop_last=True로 설정하면 batch_size보다 작을 경우 
        # 훈련 손실이 갑자기 높아지는 것을 피하기 위해 마지막 배치를 삭제함
        drop_last=drop_last,
        
        # 전처리에 사용할 CPU 프로세서 개수
        num_workers=num_workers
    )

    return dataloader

#### GPTDatasetV1 클래스와 create_dataloader_v1 함수 동작 방식

In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f :
    raw_text = f.read()
    
print(raw_text[:100])
    
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)

In [ ]:
# dataloader를 파이썬 반복자 (iterable)로 변환한 다음,
# 파이썬 내장 next() 함수로 다음 원소 추출
data_iter = iter(dataloader)
first_batch = next(data_iter)
second_batch = next(data_iter)
print("첫 번째 배치의 입/출력 시퀀스\n:", first_batch)
print("두 번째 배치의 입/출력 시퀀스\n:", second_batch)

#### 위 코드 결과
    - first_batch 변수: tensor 2개
        - 첫번째 텐서: 입력 토큰 ID
        - 두번째 텐서: 타겟 토큰 ID
    - batch_size=4는 매우 작음
        - 일반적으로 LLM 훈련 시 batch_size=256 사용    
        - batch_size가 너무 작으면 model update 과정에 noise가 많이 낌

#### stride=1 의미
    - 첫 번째 배치, 두 번째 배치 비교
        - 두 번째 배치의 토큰 ID가 하나씩 밀려있음
    - stride : sliding window가 배치에 걸쳐 입력 위를 이동하는 크기

#### batch_size가 1보다 클 경우 dataloader로 sampling 하는 방법

In [ ]:
# stride=4로 설정하여 max_length=4인 시퀀스가 겹치지 않으면서 한 토큰도 건너뛰지 않도록 함
# batch 사이에 중첩이 있으면 과대적합이 발생할 수 있음
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=4, stride=4,
    shuffle=False
)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("입력 시퀀스\n:", inputs)
print("출력 시퀀스\n:", targets)

## 2.6. Token Embedding 
#### 토큰 ID → Embedding Vector
    - 연속적인 벡터 표현인 임베딩이 필수적임

#### ex) 6개의 단어로 구성된 작은 어휘사전 가정 + 크기 3짜리 임베딩 만들기
    - GPT-3의 임베딩 크기는 12,288차원

In [ ]:
input_ids = torch.tensor([2, 3, 5, 1])
vocab_size = 6
output_dim = 3

##### 임베딩 층의 가중치 행렬 : 작고 랜덤한 값을 담고 있음
    - 이 값 역시 LLM 최적화의 일부임

In [ ]:
# random seed 고정 -> 모델 훈련 시 재현 가능한 결과를 얻기 위해 랜덤 시드를 고정
torch.manual_seed(123)

embedding_layer = torch.nn.Embedding(num_embeddings=vocab_size, embedding_dim=output_dim)
print("embedding_layer.weight\n:", embedding_layer.weight)

In [ ]:
print(embedding_layer(torch.tensor([3])))
print('')
print(embedding_layer(input_ids)) # input_ids = torch.tensor([2, 3, 5, 1])

#### 출력된 행렬의 각 행은 임베딩 가중치 행렬의 룩업(lookup) 연산을 통해 구할 수 있음

# 7. 단어 위치 Encoding
#### 토큰 임베딩은 원칙적으로 LLM의 입력으로 적합
    - But LLM의 단점 중 하나 : 시퀀스 안의 토큰 위치 or 순서에 대한 개념이 self-attention mechanism에 없다는 것
    - 입력 시퀀스 안에 토큰 ID의 위치에 상관없이 동일한 토큰 ID를 항상 동일한 벡터 표현에 매핑함

#### 절대 위치 임베딩
    - 시퀀스의 특정 위치에 직접 연관
    - 입력 시퀀스의 각 위치에 대해 고유한 임베딩이 토큰 임베딩에 더해져 정확한 위치 정보 추가
    - ex) 
        - 첫 번째 토큰 : 특정 위치 임베딩 사용
        - 두 번째 토큰 : 또 다른 고유한 위치 임베딩 사용

#### 상대 위치 임베딩
    - 토큰의 절대 위치에 초점을 맞추는 대신 상대적인 위치 or 토큰 사이의 거리를 강조함
    - 모델이 길이가 다른 시퀀스에도 더 잘 일반화될 수 있음

#### OpenAI의 GPT 모델은 훈련 과정에서 최적화되는 절대 위치 임베딩을 사용
    - 더 현실적이고 유용한 임베딩 크기를 선택하여 input 토큰을 256차원의 벡터 표현으로 인코딩
    - 토큰ID를 50,257 크기의 어휘사전을 가진 BPE 토크나이저로 만들었다고 가정
    - token_embedding_layer를 사용해 dataloader를 통해 샘플링한 각 배치에 있는 토큰 ID를 256차원 벡터로 임베딩함
    - batch_size가 8, 4개의 토큰씩 들어있다면 만들어진 벡터는 8x4x256 텐서가 될 것임

In [ ]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(num_embeddings=vocab_size, embedding_dim=output_dim)

In [ ]:
# 토큰 ID 텐서 차원: (batch_size, sequence_length) -> (8, 4)
# 배치에 4개의 토큰을 가진 텍스트 샘플 8개가 들어있다는 의미

max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length, stride=max_length,
    shuffle=False
)   
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("토큰 ID:\n", inputs)
print("\n입력 크기:\n", inputs.shape)

In [ ]:
# 8x4x256 차원의 텐서는 각 토큰ID가 256차원 벡터로 임베딩 되었다는 것을 보여줌
token_embeddings = token_embedding_layer(inputs)
print("토큰 임베딩:\n", token_embeddings.shape)

#### pos_embeddings의 입력은 일반적으로 placeholder 벡터인 torch.arange(context_length)임
    - context_length : LLM이 지원하는 입력 크기를 나타내는 변수
    - 아래 코드를 보면 위치 임베딩 벡터가 4개의 256차원 벡터로 구성되는 걸 볼 수 있음
    - pytorch는 4 x 256 차원의 pos_embeddings 텐서를 배치에 있는 4 x 256
      차원의 토큰 임베딩 텐서 8개에 각각 더함

In [ ]:
# GPT 모델의 절대 임베딩 방법에서는 token_embedding_layer와 동일한 임베딩
# 차원을 가지는 또 다른 임베딩 층을 만들면 됨

context_length = max_length
pos_embedding_layer = torch.nn.Embedding(num_embeddings=context_length, embedding_dim=output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print("위치 임베딩:\n", pos_embeddings.shape)

In [ ]:
input_embeddings = token_embeddings + pos_embeddings
print("입력 임베딩:\n", input_embeddings)
print("")
print("입력 임베딩 크기:\n", input_embeddings.shape)